In [ ]:
import xarray as xr
import rasterio
from shapely.geometry import Polygon
import geopandas as gpd
import numpy as np
import os
import pandas as pd
from utils.data_prep import h3_grid as h3
from utils.ahp import Ahp_calc
from utils.geo_score_converter import GeoIntervalScorer

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [72]:
# wind_speed
path = '/home/jta/Documentos/articulo_mcdm_lca/data/TEC-001/Velocidad_100m_patched.tif'
ds_wind = rasterio.open(path)
crs = 'EPSG:9377'
# Se corre malla resolucion 6 por temas de computo
resolution = 6

h3_module = h3(
    raster=ds_wind,
    resolution=6,
    crs=crs
)

path = 'seeds/codigo_tipos(in).csv'
df_data = pd.read_csv(path,sep=';',encoding='latin-1')
df_data = df_data.drop(['Criterio'], axis = 1)
# df_data = df_data[~df_data['Subcriterio'].isna()]

## Preparación Malla

In [73]:
path = '/home/jta/Documentos/articulo_mcdm_lca/data'
files = os.listdir(path)
# Mantiene solo archivos que se encuentran en los datos
mask = df_data['Codigo'].isin(files)
df_data = df_data[mask].reset_index(drop=True)

In [74]:
malla_generada =h3_module.vars_ahp(path=path,df_data=df_data)

EPSG:9377
procesando archivo AMB-001-AR
procesando archivo AMB-003-A
procesando archivo AMB-004-A
procesando archivo AMB-005-A
procesando archivo AMB-006-A


KeyboardInterrupt: 

In [ ]:
# Actualiza las unidades de los datos
malla_generada['TEC-023'] = malla_generada['TEC-023']/1000
malla_generada['TEC-032'] = malla_generada['TEC-032']/1000 

Conversión de los datos a los intervalos. A partir de la malla generada. Los datos de litología se convierten a partir de la información suministrada por Manuela

In [ ]:
# Primero convierte los datos de litología
def score_litologia(litol):

    #En caso que halla un valor nan
    if pd.isna(litol):
        score = 0
    else:
        # Lectura de archivo de scores
        path_scores = 'seeds/score_litologia.csv'
        df = pd.read_csv(path_scores)
        # Extrae solo la litologia de interes
        mask = df['Descripcio'] == litol
        df_slice = df[mask]
        # Extra el score asociado
        score = df_slice.iloc[0,1]
    return score

# Actualizacion
malla_generada['AMB-028-AT'] = malla_generada['AMB-028-AT'].apply(score_litologia) 


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/home/jta/miniconda3/envs/cds/lib/python3.14/site-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_21491/700173494.py", line 19, in <module>
    malla_generada['AMB-028-AT'] = malla_generada['AMB-028-AT'].apply(score_litologia)
                                   ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/home/jta/miniconda3/envs/cds/lib/python3.14/site-packages/pandas/core/series.py", line 4943, in apply
    ).apply()
      ~~~~~^^
  File "/home/jta/miniconda3/envs/cds/lib/python3.14/site-packages/pandas/core/apply.py", line 1422, in apply
    return self.apply_standard()
           ~~~~~~~~~~~~~~~~~~~^^
  File "/home/jta/miniconda3/envs/cds/lib/python3.14/site-packages/pandas/core/apply.py", line 1502, in apply_standard
    mapped = obj._map_values(
        mapper=curried, na_ac

## Estimacion de intervalos

In [148]:
malla = gpd.read_file('malla_articulo')

In [149]:
# Variables a convertir
mask = df_data['Tipo'] == 'excluyente'
df_vars = df_data[~mask]
convertir = list(df_data['Codigo'])

In [239]:
convertir.remove('AMB-040-A')

In [240]:
trans = GeoIntervalScorer('seeds/intervalos.csv','seeds/codigo_tipos(in).csv')
temp = trans.transform(malla_generada, columns=convertir)

/home/jta/Documentos/articulo_mcdm_lca/ahpAmbiental/utils/geo_score_converter.py:79: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  


## Generación de matrices de pesos

In [221]:
def estimacion_pesos(vars,matrix) -> dict:
    w = (matrix / matrix.sum(0)).mean(1)
    pesos_vars = dict(zip(vars, w))

    return pesos_vars

In [ ]:
ahp = Ahp_calc()

In [ ]:
CODIGOS_BIOTICAS = [
    "AMB-042-A",  # Murciélagos IUCN
    "AMB-005-A",  # Especies endémicas
    "AMB-017-A",  # Aves IUCN
    "AMB-003-A"   # Especies migratorias
]

CODIGOS_PROTEGIDAS = [
    "AMB-012-A",  # Humedales RAMSAR
    "AMB-033-A"   # KBAs / áreas clave de biodiversidad
]

# ==========================================
# TENSORES (AGRUPACIÓN DE LOS 5 PERFILES)
# ==========================================

# 1. Variable: tensor_bioticas
# Dimensiones: (5 perfiles, 4 filas, 4 columnas)
tensor_bioticas = np.array([
    # Perfil 1
    [[1.0,  1/4,  1/2,  2.0],
     [4.0,  1.0,  2.0,  6.0],
     [2.0,  1/2,  1.0,  3.0],
     [1/2,  1/6,  1/3,  1.0]],
     
    # Perfil 2
    [[1.0,  1/3,  1.0,  3.0],
     [3.0,  1.0,  3.0,  5.0],
     [1.0,  1/3,  1.0,  3.0],
     [1/3,  1/5,  1/3,  1.0]],
     
    # Perfil 3
    [[1.0,  1/3,  1/2,  2.0],
     [3.0,  1.0,  2.0,  4.0],
     [2.0,  1/2,  1.0,  3.0],
     [0.5,  1/4,  1/3,  1.0]],
     
    # Perfil 4
    [[1.0,  1/3,  1/2,  2.0],
     [3.0,  1.0,  2.0,  5.0],
     [2.0,  1/2,  1.0,  3.0],
     [1/2,  1/5,  1/3,  1.0]],
     
    # Perfil 5
    [[1.0,  1/3,  1.0,  3.0],
     [3.0,  1.0,  3.0,  5.0],
     [1.0,  1/3,  1.0,  3.0],
     [1/3,  1/5,  1/3,  1.0]]
])

w = ahp.calculate_weights(tensor_bioticas)
pesos_bioticas = dict(zip(CODIGOS_BIOTICAS, w))
bioticas = temp[list(pesos_bioticas.keys())] * pesos_bioticas.values()
bioticas = bioticas.fillna(0)
especies = bioticas.sum(axis=1)


In [263]:
# 2. Variable: tensor_protegidas
# Dimensiones: (5 perfiles, 2 filas, 2 columnas)
tensor_protegidas = np.array([
    # Perfil 1
    [[1.0, 1/3],
     [3.0, 1.0]],
     
    # Perfil 2
    [[1.0, 1/3],
     [3.0, 1.0]],
     
    # Perfil 3
    [[1.0, 0.5],
     [2.0, 1.0]],
     
    # Perfil 4
    [[1.0, 1/2],
     [2.0, 1.0]],
     
    # Perfil 5
    [[1.0, 3.0],
     [1/3, 1.0]]
])
w = ahp.calculate_weights(tensor_protegidas)
pesos_protegidas = dict(zip(CODIGOS_PROTEGIDAS, w))
# Multiplicacion pesos
protegidas = temp[list(pesos_protegidas.keys())] * pesos_protegidas.values()
# eliminar NaNs
protegidas = protegidas.fillna(0)
# Paso a la parte de subcriterio
a_protegidas = protegidas.sum(axis=1)

In [ ]:
GRUPOS_SUB_BIOTICAS = [
    "Especies Bióticas",
    "Áreas Protegidas",
]

matrix_sub_bioticas = np.array([
    [1,   1/3],
    [3,   1],
])

w = ahp.calculate_weights([matrix_sub_bioticas])
pesos_sub_bioticas = dict(zip(GRUPOS_SUB_BIOTICAS, w))

var_biotica = especies * pesos_sub_bioticas['Especies Bióticas'] + a_protegidas * pesos_sub_bioticas['Áreas Protegidas']

In [268]:
CODIGOS_GEOLOGIA = [
    "AMB-006-A",    # Uso / cobertura de suelo (Compatibilidad territorial)
    "AMB-007-AT",   # Pendiente del terreno (Riesgo de remoción en masa)
    "AMB-028-AT",   # Litología (Capacidad de carga de la cimentación)
    "AMB-008-AT",   # Elevación (Accesibilidad y costos logísticos)
]

# Matriz AHP (7x7) basada en la escala de Saaty [4, 6]
# Representa la importancia relativa de las filas sobre las columnas
# ==========================================
# 2. CATEGORÍA: CONDICIONES GEOLÓGICAS Y GEOTÉCNICAS
# ==========================================
# Matrices 4x4 de cada perfil
m_geo_1 = np.array([
    [1.0,     2.0,     5.0,     7.0],
    [1.0/2.0, 1.0,     3.0,     5.0],
    [1.0/5.0, 1.0/3.0, 1.0,     3.0],
    [1.0/7.0, 1.0/5.0, 1.0/3.0, 1.0]
])

m_geo_2 = np.array([
    [1.0,     1.0/3.0, 1.0/5.0, 3.0],
    [3.0,     1.0,     1.0/3.0, 5.0],
    [5.0,     3.0,     1.0,     7.0],
    [1.0/3.0, 1.0/5.0, 1.0/7.0, 1.0]
])

m_geo_3 = np.array([
    [1.0,   1/4.0, 1/5.0, 2.0],
    [4.0,   1.0,   1/2.0, 6.0],
    [5.0,   2.0,   1.0,   7.0],
    [1/2.0, 1/6.0, 1/7.0, 1.0]
])

m_geo_4 = np.array([
    [1.0,  1/4,  1/5,  2.0],
    [4.0,  1.0,  1/2,  5.0],
    [5.0,  2.0,  1.0,  7.0],
    [1/2,  1/5,  1/7,  1.0]
])

m_geo_5 = np.array([
    [1.0,  1/4,  1/5,  2.0],
    [4.0,  1.0,  1/2,  5.0],
    [5.0,  2.0,  1.0,  6.0],
    [0.5,  1/5,  1/6,  1.0]
])

# Tensor de matrices geotécnicas (Forma: 5 x 4 x 4)
tensor_geotecnia = np.array([m_geo_1, m_geo_2, m_geo_3, m_geo_4, m_geo_5])

w = ahp.calculate_weights(tensor_geotecnia)
pesos_geologia= dict(zip(CODIGOS_GEOLOGIA, w))


In [269]:
cols = list(pesos_geologia.keys())
temp[cols] * pesos_geologia.values()
geologia = temp[cols] * pesos_geologia.values()
# Remover nans
geologia = geologia.fillna(0).sum(axis=1)
amenazas = temp['AMB-040-A'].fillna(0)

In [ ]:
# Generar abioticas

In [272]:
variables = [
    "TEC-001",
    "TEC-023",
    "TEC-032"
]

ahp_matrix = np.array([
    [1.000, 5.000, 3.000],
    [1/5,   1.000, 1/3],
    [1/3,   3.000, 1.000]
])

ahp_matrix_tec = np.array([
    [1.0,     5.0,     7.0],
    [1.0/5.0, 1.0,     3.0],
    [1.0/7.0, 1.0/3.0, 1.0]
])

matriz_tecnica_operativa = np.array([
    [1,      5,    3],
    [1/5,    1,  1/2],
    [1/3,    2,    1]
])

A = np.array([
    [1.00, 5.00, 4.00],
    [0.20, 1.00, 0.50],
    [0.25, 2.00, 1.00]
])

ahp_matrix_technical = np.array([
    [1.0,       1.0/3.0,   3.0],
    [3.0,       1.0,       7.0],
    [1.0/3.0,   1.0/7.0,   1.0]
])

# 1. Crear el Tensor en NumPy (Apilar las matrices)
tensor_tecnico = np.stack([
    ahp_matrix, 
    ahp_matrix_tec, 
    matriz_tecnica_operativa, 
    A, 
    ahp_matrix_technical
])

In [273]:
w = ahp.calculate_weights(tensor_tecnico)
pesos_tecnico = dict(zip(variables, w))
# Multiplicacion pesos
tecnico = temp[list(pesos_tecnico.keys())] * pesos_tecnico.values()
# eliminar NaNs
tecnico = tecnico.fillna(0)
# Paso a la parte de subcriterio
arreglo_tecnico = tecnico.sum(axis=1)

## Nivel 2 AHP

In [ ]:
import numpy as np

# Definición de la matriz de comparación pareada (Saaty 1-9)
# Orden de variables: [ESP, AP]

matriz_1 = np.array([
    [1.0000, 1/3.0],
    [3.0000, 1.0000]
])

matriz_2 = np.array([
    [1.0, 3.0],
    [1.0 / 3.0, 1.0]
])

matriz_3 = np.array([
    [1.0, 1.0 / 3.0],
    [3.0, 1.0]
])

matriz_4 = np.array([
    [1.0, 1.0 / 3.0],
    [3.0, 1.0]
])

matriz_5 = np.array([
    [1.000, 1.0 / 3.0],
    [3.000, 1.000]
])

# Creación del tensor combinando todas las matrices con orden [ESP, AP]
tensor_matrices = np.array([matriz_1, matriz_2, matriz_3, matriz_4, matriz_5])

print("Forma del tensor:", tensor_matrices.shape)
print("Contenido del tensor:\n", tensor_matrices)

In [ ]:
import numpy as np

# Variables evaluadas: 
# # Índice 0: Amenazas (AME) 
# # Índice 1: Condiciones Geológicas y geotécnicas (CGG)

# 1. Definir las matrices individuales
m1 = np.array([
    [1.0, 2.0],
    [0.5, 1.0]
])

m2 = np.array([
    [1.0, 1.0/3.0],
    [3.0, 1.0]
])

m3 = np.array([
    [1.0, 1/2],
    [2.0, 1.0]
])

m4 = np.array([
    [1.0, 3.0],      
    [1.0/3.0, 1.0]   
])

m5 = np.array([
    [1.0, 1/2],  
    [2.0, 1.0]   
])

# 2. Combinarlas en un tensor 3D usando np.array o np.stack
tensor_ahp = np.array([m1, m2, m3, m4, m5])

# Mostrar el resultado y las dimensiones
print("Dimensiones del tensor:", tensor_ahp.shape)
# Salida esperada: (5, 2, 2) -> (número de matrices, filas, columnas)